# Phase 5: Human-in-the-Loop Validation

**Interpretability of Portuguese Language Models via Sparse Autoencoders**

---

**Goal:** Validate the 150 selected features with a hybrid LLM + human protocol.

**What this notebook does:**
1. Loads model, SAE, and selected features (from Phase 3/4)
2. Extracts **top-20 activating examples** per feature from the corpora
3. Uses the **Claude (Anthropic) API** to generate automatic descriptions of each feature
4. Exports a **validation template** (JSON) for manual review
5. After manual review, imports and computes **agreement statistics**

**Validation protocol (per feature):**
1. The LLM (Claude) generates a description based on the top-20 examples
2. The researcher examines 10-20 examples and rates the description (0-3 scale)
3. If in disagreement → records their own interpretation
4. Final classification = human judgment

**Prerequisites:**
- GPU with ≥16GB VRAM (Colab T4)
- Phase 3 checkpoints on Google Drive
- Anthropic API key (obtained at [console.anthropic.com](https://console.anthropic.com))
- Estimated runtime: ~40 min (example extraction + description generation)

In [ ]:
!pip install sae-lens transformer-lens anthropic -q
print()
print("Installation complete. Restart the runtime before continuing:")
print("  Runtime → Restart session (ou Ctrl+M .)")
print("  Then skip this cell and run from the next one.")

In [ ]:
import torch
import numpy as np
import json
import os
import time
from tqdm.auto import tqdm

if torch.cuda.is_available():
    device = 'cuda'
    gpu_name = torch.cuda.get_device_name()
    gpu_mem = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f'GPU: {gpu_name} ({gpu_mem:.1f} GB)')
elif hasattr(torch.backends, 'mps') and torch.backends.mps.is_available():
    device = 'mps'
    print('Usando Apple Silicon (MPS)')
else:
    device = 'cpu'
    print('No GPU detected.')

print(f'Device: {device}')
print(f'PyTorch: {torch.__version__}')

In [ ]:
from huggingface_hub import login
login()

In [ ]:
from sae_lens import SAE, HookedSAETransformer

LAYER = 13
SAE_RELEASE = "gemma-scope-2b-pt-res-canonical"
SAE_ID = f"layer_{LAYER}/width_16k/canonical"

print("Loading Gemma 2 2B...")
model = HookedSAETransformer.from_pretrained_no_processing(
    "gemma-2-2b",
    device=device,
    dtype=torch.float16,
)
print(f"Model: {model.cfg.model_name} | Layers: {model.cfg.n_layers} | d_model: {model.cfg.d_model}")

print()
print(f"Loading SAE: {SAE_ID}...")
sae, cfg_dict, sparsity = SAE.from_pretrained(
    release=SAE_RELEASE,
    sae_id=SAE_ID,
    device=device,
)

HOOK_NAME = f"blocks.{LAYER}.hook_resid_post"
print(f"SAE: {sae.cfg.d_sae} features | d_in: {sae.cfg.d_in} | hook: {HOOK_NAME}")

In [ ]:
from google.colab import drive
drive.mount("/content/drive", force_remount=True)

In [ ]:
from google.colab import drive

SAVE_DIR = "./checkpoints/"  # adjust to your preferred path (e.g., ./checkpoints/ for Colab)
os.makedirs(SAVE_DIR, exist_ok=True)

MIN_ACTS = 100
LSI_THRESHOLD = 0.3
N_SELECT = 50

phase3_path = SAVE_DIR + "phase3_results.pt"
if os.path.exists(phase3_path):
    print(f"Loading phase3_results.pt...")
    phase3 = torch.load(phase3_path, map_location="cpu")
    selected_pt = phase3["selected"]["pt"]
    selected_cross = phase3["selected"]["cross"]
    selected_en = phase3["selected"]["en"]
    all_selected = phase3["selected"]["all"]
    lsi_combined = phase3["lsi"]["combined"]
    lsi_wiki = phase3["lsi"]["wiki"]
    lsi_web = phase3["lsi"]["web"]
    pt_both = phase3["masks"]["pt_both"]
    has_examples = "examples" in phase3 and phase3["examples"]["pt"]
    if has_examples:
        print("  Phase 3 examples found! Skipping extraction.")
        examples_pt = phase3["examples"]["pt"]
        examples_cross = phase3["examples"]["cross"]
        examples_en = phase3["examples"]["en"]
else:
    print("Recomputing selection from individual checkpoints...")
    has_examples = False

    ckpt_files = ["stats_wiki_pt.pt", "stats_wiki_en.pt", "stats_mc4_pt.pt", "stats_c4_en.pt"]
    stats = {}
    for fname in ckpt_files:
        path = SAVE_DIR + fname
        if not os.path.exists(path):
            path = fname
        key = fname.replace("stats_", "").replace(".pt", "")
        stats[key] = torch.load(path, map_location="cpu")
        print(f"  Loaded: {fname}")

    freq_wiki_pt = stats["wiki_pt"]["counts"] / stats["wiki_pt"]["total_tokens"]
    freq_wiki_en = stats["wiki_en"]["counts"] / stats["wiki_en"]["total_tokens"]
    freq_mc4_pt = stats["mc4_pt"]["counts"] / stats["mc4_pt"]["total_tokens"]
    freq_c4_en = stats["c4_en"]["counts"] / stats["c4_en"]["total_tokens"]

    lsi_wiki = (freq_wiki_pt - freq_wiki_en) / (freq_wiki_pt + freq_wiki_en + 1e-10)
    lsi_web = (freq_mc4_pt - freq_c4_en) / (freq_mc4_pt + freq_c4_en + 1e-10)
    lsi_combined = (lsi_wiki + lsi_web) / 2

    total_counts = stats["wiki_pt"]["counts"] + stats["wiki_en"]["counts"] + stats["mc4_pt"]["counts"] + stats["c4_en"]["counts"]
    active = total_counts >= MIN_ACTS

    pt_wiki_mask = (lsi_wiki > LSI_THRESHOLD) & active
    pt_web_mask = (lsi_web > LSI_THRESHOLD) & active
    pt_both = pt_wiki_mask & pt_web_mask

    lsi_pt_rank = lsi_combined.clone()
    lsi_pt_rank[~active] = -2
    _, top_pt_idx = lsi_pt_rank.topk(N_SELECT)
    selected_pt = top_pt_idx.tolist()

    lsi_en_rank = lsi_combined.clone()
    lsi_en_rank[~active] = 2
    _, top_en_idx = lsi_en_rank.topk(N_SELECT, largest=False)
    selected_en = top_en_idx.tolist()

    total_freq = freq_wiki_pt + freq_wiki_en + freq_mc4_pt + freq_c4_en
    cross_score = lsi_combined.abs().clone()
    cross_score[~active] = 999
    min_cross_freq = total_freq[active].median()
    cross_score[total_freq < min_cross_freq] = 999
    _, top_cross_idx = cross_score.topk(N_SELECT, largest=False)
    selected_cross = top_cross_idx.tolist()

    all_selected = selected_pt + selected_cross + selected_en

print(f"\nFeatures: {len(selected_pt)} PT + {len(selected_cross)} cross + {len(selected_en)} EN = {len(all_selected)}")
print(f"Preloaded examples: {'Yes' if has_examples else 'No (will be extracted)'}")

---
# Activating Example Extraction

For each of the 150 features, we extract the **top-20 texts** that most activate the feature.
- PT and cross-lingual features → examples from **Wikipedia PT**
- EN features → examples from **Wikipedia EN**

These examples serve as the basis for the automatic (LLM) description and for manual validation.

In [ ]:
if not has_examples:
    from datasets import load_dataset

    tokenizer = model.tokenizer
    SEQ_LEN = 256
    N_TOKENS_EXAMPLES = 2_000_000

    def collect_tokens(dataset, n_tokens, text_field="text", desc="Tokenizando"):
        all_ids = []
        n_articles = 0
        for article in tqdm(dataset, desc=desc):
            text = article[text_field]
            if not text or len(text) < 50:
                continue
            ids = tokenizer.encode(text, add_special_tokens=False)
            all_ids.extend(ids)
            n_articles += 1
            if len(all_ids) >= n_tokens:
                break
        all_ids = all_ids[:n_tokens]
        n_seqs = len(all_ids) // SEQ_LEN
        tokens = torch.tensor(all_ids[:n_seqs * SEQ_LEN], dtype=torch.long).reshape(n_seqs, SEQ_LEN)
        print(f"  {n_articles} texts -> {tokens.numel():,} tokens ({tokens.shape[0]} seqs)")
        return tokens

    print("Tokenizing Wikipedia PT (for examples)...")
    wiki_pt = load_dataset("wikimedia/wikipedia", "20231101.pt", split="train", streaming=True)
    tokens_wiki_pt = collect_tokens(wiki_pt, N_TOKENS_EXAMPLES, desc="Wiki PT")

    print("\nTokenizing Wikipedia EN (for examples)...")
    wiki_en = load_dataset("wikimedia/wikipedia", "20231101.en", split="train", streaming=True)
    tokens_wiki_en = collect_tokens(wiki_en, N_TOKENS_EXAMPLES, desc="Wiki EN")

    print("\nTokenization complete!")
else:
    print("Examples already loaded from Phase 3. Skipping tokenization.")

In [ ]:
if not has_examples:
    BATCH_SIZE = 8
    N_EXAMPLES = 20

    @torch.no_grad()
    def find_top_examples(model, sae, tokens, feature_indices,
                          n_examples=N_EXAMPLES, batch_size=BATCH_SIZE, max_batches=500):
        hook = HOOK_NAME
        feat_list = list(feature_indices)
        results = {f: [] for f in feat_list}
        tok = model.tokenizer

        actual_batches = min(max_batches, (len(tokens) + batch_size - 1) // batch_size)
        for i in tqdm(range(0, actual_batches * batch_size, batch_size),
                      desc="Searching for examples", total=actual_batches):
            if i >= len(tokens):
                break
            batch = tokens[i:i+batch_size].to(device)
            _, cache = model.run_with_cache(batch, names_filter=lambda n: n == hook)
            acts = cache[hook]
            feat_acts = sae.encode(acts)

            for f_idx in feat_list:
                vals = feat_acts[:, :, f_idx]
                for b in range(batch.shape[0]):
                    max_val, max_pos = vals[b].max(dim=0)
                    if max_val.item() > 0:
                        pos = max_pos.item()
                        start = max(0, pos - 10)
                        end = min(batch.shape[1], pos + 15)
                        ctx = tok.decode(batch[b, start:end].tolist())
                        results[f_idx].append({"act": max_val.item(), "context": ctx})

            del cache, acts, feat_acts
            if device == "cuda":
                torch.cuda.empty_cache()

        for f in results:
            results[f].sort(key=lambda x: x["act"], reverse=True)
            results[f] = results[f][:n_examples]
        return results

    print("Extracting examples for PT-specific features (Wiki PT)...")
    examples_pt = find_top_examples(model, sae, tokens_wiki_pt, selected_pt)

    print("\nExtracting examples for cross-lingual features (Wiki PT)...")
    examples_cross = find_top_examples(model, sae, tokens_wiki_pt, selected_cross)

    print("\nExtracting examples for EN-specific features (Wiki EN)...")
    examples_en = find_top_examples(model, sae, tokens_wiki_en, selected_en)

    examples_ckpt = {
        "pt": examples_pt,
        "cross": examples_cross,
        "en": examples_en,
    }
    torch.save(examples_ckpt, SAVE_DIR + "phase5_examples.pt")
    print(f"\nExamples saved to: {SAVE_DIR}phase5_examples.pt")
else:
    print("Examples already available. Skipping extraction.")

print(f"\nExamples extracted:")
print(f"  PT: {sum(len(v) for v in examples_pt.values())} examples for {len(examples_pt)} features")
print(f"  Cross: {sum(len(v) for v in examples_cross.values())} examples for {len(examples_cross)} features")
print(f"  EN: {sum(len(v) for v in examples_en.values())} examples for {len(examples_en)} features")

---
# Generating Descriptions via LLM (Claude/Anthropic)

For each feature, we send the top-20 activating examples to **Claude (Anthropic)** and ask for a concise description of what the feature detects.

**Get an API key:**
1. Go to [console.anthropic.com](https://console.anthropic.com)
2. Create an API key in the account settings
3. Paste the key in the cell below

The generated description will later be rated by the researcher (0-3 scale).

In [ ]:
import anthropic
from google.colab import userdata

try:
    ANTHROPIC_API_KEY = userdata.get('ANTHROPIC_API_KEY')
    print("Chave API carregada dos secrets do Colab.")
except Exception:
    ANTHROPIC_API_KEY = input("Cole sua chave API da Anthropic: ").strip()

client = anthropic.Anthropic(api_key=ANTHROPIC_API_KEY)

test = client.messages.create(
    model="claude-opus-4-20250514",
    max_tokens=50,
    messages=[{"role": "user", "content": "Responda apenas 'OK' se estiver funcionando."}],
)
print(f"Teste Claude: {test.content[0].text.strip()}")
print("API configurada com sucesso!")

In [ ]:
def get_category(feat_id):
    if feat_id in selected_pt:
        return "PT-specific"
    elif feat_id in selected_en:
        return "EN-specific"
    else:
        return "Cross-lingual"

def get_examples(feat_id):
    if feat_id in examples_pt:
        return examples_pt[feat_id]
    elif feat_id in examples_cross:
        return examples_cross[feat_id]
    elif feat_id in examples_en:
        return examples_en[feat_id]
    return []

def build_prompt(feat_id, examples, lsi_val, category):
    examples_text = ""
    for i, ex in enumerate(examples[:20]):
        examples_text += '{0}. [activation: {1:.2f}] "{2}"\n'.format(i+1, ex["act"], ex["context"])

    prompt = (
        "You are analyzing features extracted from a Sparse Autoencoder (SAE) "
        "applied to the Gemma 2 2B language model at layer 13 (residual stream).\n\n"
        "Below are the top activating text segments for a specific feature. Each segment "
        "shows the text context around the token that most strongly activates this feature, "
        "along with the activation strength.\n\n"
        "Feature ID: {feat_id}\n"
        "Language Specificity Index (LSI): {lsi_val:+.4f} "
        "(range: -1.0 = English-only, +1.0 = Portuguese-only, 0.0 = cross-lingual)\n"
        "Category: {category}\n\n"
        "Top activating examples (sorted by activation strength):\n"
        "{examples_text}\n"
        "Based on these examples, provide:\n"
        "1. A concise description (1-2 sentences) of what this feature detects or represents.\n"
        "2. If applicable, identify the specific linguistic phenomenon "
        "(e.g., grammatical gender, crase, clitic pronouns, verb conjugation, prepositions, contractions, etc.).\n"
        "3. Rate your confidence: HIGH (clear pattern), MEDIUM (likely pattern), or LOW (unclear).\n\n"
        "Respond in this exact format:\n"
        "DESCRIPTION: [your description]\n"
        "PHENOMENON: [linguistic phenomenon or 'general' if not specific]\n"
        "CONFIDENCE: [HIGH/MEDIUM/LOW]"
    ).format(feat_id=feat_id, lsi_val=lsi_val, category=category, examples_text=examples_text)
    return prompt

ckpt_path = SAVE_DIR + "phase5_descriptions.json"
if os.path.exists(ckpt_path):
    with open(ckpt_path, "r") as f:
        descriptions = json.load(f)
    print(f"Loaded {len(descriptions)} descriptions from checkpoint.")
    remaining = [f for f in all_selected if str(f) not in descriptions]
else:
    descriptions = {}
    remaining = list(all_selected)

print(f"Features to process: {len(remaining)} (de {len(all_selected)} total)")

errors = []
for i, feat_id in enumerate(remaining):
    examples = get_examples(feat_id)
    if not examples:
        descriptions[str(feat_id)] = {
            "description": "No examples available",
            "phenomenon": "unknown",
            "confidence": "LOW",
            "raw": "",
        }
        continue

    lsi_val = lsi_combined[feat_id].item()
    category = get_category(feat_id)
    prompt = build_prompt(feat_id, examples, lsi_val, category)

    try:
        response = client.messages.create(
              model="claude-opus-4-20250514",
              max_tokens=300,
              messages=[{"role": "user", "content": prompt}],
          )
        text = response.content[0].text.strip()
        desc = ""
        phenom = ""
        conf = ""
        for line in text.split("\n"):
            line = line.strip()
            if line.startswith("DESCRIPTION:"):
                desc = line[len("DESCRIPTION:"):].strip()
            elif line.startswith("PHENOMENON:"):
                phenom = line[len("PHENOMENON:"):].strip()
            elif line.startswith("CONFIDENCE:"):
                conf = line[len("CONFIDENCE:"):].strip()

        descriptions[str(feat_id)] = {
            "description": desc or text,
            "phenomenon": phenom or "unknown",
            "confidence": conf or "MEDIUM",
            "raw": text,
        }

        if (i + 1) % 10 == 0:
            with open(ckpt_path, "w") as f:
                json.dump(descriptions, f, ensure_ascii=False, indent=2)
            print(f"  [{i+1}/{len(remaining)}] Checkpoint saved")

    except Exception as e:
        errors.append((feat_id, str(e)))
        print(f"  Error in feature {feat_id}: {e}")

    time.sleep(1.5)

with open(ckpt_path, "w") as f:
    json.dump(descriptions, f, ensure_ascii=False, indent=2)

print(f"\nDescriptions generated: {len(descriptions)}")
if errors:
    print(f"Errors: {len(errors)}")
    for fid, err in errors[:5]:
        print(f"  Feature {fid}: {err}")
print(f"Saved to: {ckpt_path}")

In [ ]:
print("=" * 90)
print("GENERATED DESCRIPTIONS — SAMPLE (top-10 PT-specific)")
print("=" * 90)

for rank, feat_id in enumerate(selected_pt[:10]):
    desc_data = descriptions.get(str(feat_id), {})
    lsi_val = lsi_combined[feat_id].item()
    examples = get_examples(feat_id)

    print(f"\n{'━' * 90}")
    print(f"Feature {feat_id} | LSI={lsi_val:+.4f} | #{rank+1} PT | {desc_data.get('confidence', '?')}")
    print(f"━" * 90)
    print(f"  LLM: {desc_data.get('description', 'N/A')}")
    print(f"  Phenomenon: {desc_data.get('phenomenon', 'N/A')}")
    print(f"  Top-5 examples:")
    for ex in examples[:5]:
        print(f'    [{ex["act"]:.1f}] ...{ex["context"][:80]}...')

---
# Manual Validation Protocol

The following cell generates a **validation template** (JSON) saved to Google Drive.

**For each feature, you must fill in:**

| Field | Values | Description |
|---|---|---|
| `human_score` | 0-3 | 0=irrelevant, 1=partial, 2=good, 3=perfect |
| `human_description` | text | Your interpretation (required if score ≤ 1) |
| `human_notes` | text | Optional notes |

**Agreement scale:**
- **3 — Perfect:** The LLM description captures exactly what the feature does
- **2 — Good:** The description is correct but incomplete or imprecise
- **1 — Partial:** The description captures something but misses the main pattern
- **0 — Irrelevant:** The description is wrong or the feature is not interpretable

**Tip:** Start with the 50 PT-specific features (most important for this work). The cross-lingual and EN ones are controls.

In [ ]:
validation_template = []

for feat_id in all_selected:
    desc_data = descriptions.get(str(feat_id), {})
    examples = get_examples(feat_id)
    lsi_val = lsi_combined[feat_id].item()
    category = get_category(feat_id)

    entry = {
        "feature_id": feat_id,
        "lsi": round(lsi_val, 4),
        "category": category,
        "llm_description": desc_data.get("description", ""),
        "llm_phenomenon": desc_data.get("phenomenon", ""),
        "llm_confidence": desc_data.get("confidence", ""),
        "top_examples": [
            {"activation": round(ex["act"], 2), "context": ex["context"]}
            for ex in examples[:20]
        ],
        "human_score": None,
        "human_description": None,
        "human_notes": None,
    }
    validation_template.append(entry)

template_path = SAVE_DIR + "phase5_validation_template.json"
with open(template_path, "w", encoding="utf-8") as f:
    json.dump(validation_template, f, ensure_ascii=False, indent=2)

print(f"Validation template saved to: {template_path}")
print(f"Total features: {len(validation_template)}")
print(f"  PT-specific: {sum(1 for e in validation_template if e['category'] == 'PT-specific')}")
print(f"  Cross-lingual:  {sum(1 for e in validation_template if e['category'] == 'Cross-lingual')}")
print(f"  EN-specific: {sum(1 for e in validation_template if e['category'] == 'EN-specific')}")
print()
print("INSTRUCTIONS:")
print("  1. Open the JSON file in Drive (or download to edit)")
print("  2. For each feature, fill in: human_score (0-3)")
print("  3. If score ≤ 1, fill in human_description with your interpretation")
print("  4. After reviewing, run the next cell to import and compute statistics")

---
# Import and Analysis of the Validation

After reviewing and filling in the JSON template, run the cells below to:
1. Import the responses
2. Compute agreement statistics
3. Generate a summary for the paper

**Run the cells below ONLY after completing the manual validation.**

In [ ]:
template_path = SAVE_DIR + "phase5_validation_template.json"

with open(template_path, "r", encoding="utf-8") as f:
    validated = json.load(f)

scored = [e for e in validated if e["human_score"] is not None]
unscored = [e for e in validated if e["human_score"] is None]

print("=" * 70)
print("VALIDATION STATISTICS")
print("=" * 70)
print(f"Evaluated features:    {len(scored)}/{len(validated)}")
print(f"Pending features:    {len(unscored)}")

if not scored:
    print("\nNo feature evaluated yet. Fill in the template and run again.")
else:
    scores = [e["human_score"] for e in scored]
    mean_score = np.mean(scores)
    print(f"\nMean score:            {mean_score:.2f}")
    print(f"Score distribution:")
    for s in range(4):
        count = scores.count(s)
        pct = count / len(scores) * 100
        bar = "█" * int(pct / 2)
        print(f"  {s} {'(irrelevante)' if s==0 else '(parcial)' if s==1 else '(boa)' if s==2 else '(perfeita)':>14s}: {count:3d} ({pct:5.1f}%) {bar}")

    agreement = sum(1 for s in scores if s >= 2) / len(scores)
    print(f"\nAgreement (score ≥ 2): {agreement:.1%}")

    for cat in ["PT-specific", "Cross-lingual", "EN-specific"]:
        cat_scores = [e["human_score"] for e in scored if e["category"] == cat]
        if cat_scores:
            cat_mean = np.mean(cat_scores)
            cat_agree = sum(1 for s in cat_scores if s >= 2) / len(cat_scores)
            print(f"\n  {cat}:")
            print(f"    Evaluated: {len(cat_scores)}")
            print(f"    Mean score: {cat_mean:.2f}")
            print(f"    Agreement: {cat_agree:.1%}")

    human_descs = [e for e in scored if e.get("human_description")]
    if human_descs:
        print(f"\nAlternative human descriptions: {len(human_descs)}")
        disagreements = [e for e in scored if e["human_score"] <= 1 and e.get("human_description")]
        print(f"Disagreements with own description: {len(disagreements)}")

    confidence_scores = {}
    for e in scored:
        conf = descriptions.get(str(e["feature_id"]), {}).get("confidence", "UNKNOWN")
        if conf not in confidence_scores:
            confidence_scores[conf] = []
        confidence_scores[conf].append(e["human_score"])

    print(f"\nMean score by LLM confidence:")
    for conf in ["HIGH", "MEDIUM", "LOW"]:
        if conf in confidence_scores:
            cs = confidence_scores[conf]
            print(f"  {conf}: {np.mean(cs):.2f} (n={len(cs)})")

In [ ]:
if scored:
    print("=" * 90)
    print("CATALOG OF VALIDATED FEATURES")
    print("=" * 90)

    fmt = "{:<8} {:<10} {:<14} {:<6} {:<50}"
    print(fmt.format("Feature", "LSI", "Category", "Score", "Final Description"))
    print("-" * 90)

    for e in sorted(scored, key=lambda x: (-x["lsi"] if x["category"] == "PT-specific" else x["lsi"])):
        final_desc = e.get("human_description") or e["llm_description"]
        final_desc = final_desc[:48] + ".." if len(final_desc) > 48 else final_desc
        print(fmt.format(
            e["feature_id"],
            f'{e["lsi"]:+.4f}',
            e["category"],
            e["human_score"],
            final_desc,
        ))

In [ ]:
results_phase5 = {
    "validation": validated,
    "descriptions": descriptions,
    "statistics": {
        "total": len(validated),
        "scored": len(scored),
        "mean_score": float(np.mean([e["human_score"] for e in scored])) if scored else None,
        "agreement_rate": float(sum(1 for e in scored if e["human_score"] >= 2) / len(scored)) if scored else None,
    },
    "selected_pt": selected_pt,
    "selected_cross": selected_cross,
    "selected_en": selected_en,
}

save_path = SAVE_DIR + "phase5_validation_results.pt"
torch.save(results_phase5, save_path)
print(f"Results saved to: {save_path}")

print()
print("=" * 70)
print("GENERATED FILES (all on Drive)")
print("=" * 70)
print(f"  {SAVE_DIR}phase5_examples.pt              — activating examples")
print(f"  {SAVE_DIR}phase5_descriptions.json         — LLM descriptions")
print(f"  {SAVE_DIR}phase5_validation_template.json  — manual validation template")
print(f"  {SAVE_DIR}phase5_validation_results.pt     — final results")